# SEC 财报 · Section 细分为 Trunks / Chunks

在**已解析好的 JSON**（`parsed_filings/*.json`）基础上，将每个 section 继续细分：

- **section** → **trunks[]** → **chunks[]**
- 每个 chunk 为固定 token 数（可配置），并带可追溯引用信息 `ref`。

输出写回同目录或 `parsed_filings/chunked/`，便于后续 RAG 或检索。

---
## 步骤 1：路径与配置

输入：`parsed_filings/` 下已由 `01_simple_parse_sec_filing.ipynb` 生成的 JSON。  
输出：同目录下的 `*_chunked.json`，或 `parsed_filings/chunked/`。  
可调参数：`CHUNK_TOKEN_SIZE`（每块约 token 数）、`CHUNKS_PER_TRUNK`（每个 trunk 包含的 chunk 数）。

In [ ]:
import os
import json

# 默认：当前目录为 03_Data_Processor/US，parsed_filings 在其下
_cwd = os.getcwd()
PARSED_DIR = os.path.join(_cwd, "parsed_filings")
OUTPUT_DIR = os.path.join(PARSED_DIR, "chunked")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_TOKEN_SIZE = 256
CHUNKS_PER_TRUNK = 10

print("输入目录 (parsed_filings):", PARSED_DIR)
print("输出目录 (chunked):", OUTPUT_DIR)
print("CHUNK_TOKEN_SIZE:", CHUNK_TOKEN_SIZE, "| CHUNKS_PER_TRUNK:", CHUNKS_PER_TRUNK)

输入目录 (parsed_filings): /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/US/parsed_filings
输出目录 (chunked): /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/US/parsed_filings/chunked
CHUNK_TOKEN_SIZE: 256 | CHUNKS_PER_TRUNK: 10


---
## 步骤 2：Token 估计与按 token 切分

使用简单规则：**token ≈ 词数 × 1.35**（英文近似）。若已安装 `tiktoken`，可改为按模型 tokenizer 统计。  
将一段文本按 `CHUNK_TOKEN_SIZE` 切成多段，尽量在词边界处切断；每段带 `ref`（section、trunk 下标、chunk 下标、起止 token 估计）。

In [6]:
def estimate_tokens(text: str) -> int:
    """简单用词数估计 token 数（英文约 1 词 ≈ 1.35 token）。"""
    if not text or not text.strip():
        return 0
    return max(1, int(len(text.split()) * 1.35))


def split_into_chunks(text: str, max_tokens: int):
    """
    将 text 按约 max_tokens 一段切成多段，在词边界切断。
    返回 list of (chunk_text, start_token_est, end_token_est)。
    """
    text = (text or "").strip()
    if not text:
        return []
    words = text.split()
    # 约 1 word ≈ 1.35 tokens → 每段约 max_tokens 需要约 max_tokens/1.35 个词
    words_per_chunk = max(1, int(max_tokens / 1.35))
    out = []
    start = 0
    while start < len(words):
        end = min(start + words_per_chunk, len(words))
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)
        start_tok = int(start * 1.35)
        end_tok = int(end * 1.35)
        out.append((chunk_text, start_tok, end_tok))
        start = end
    return out


def build_section_trunks_chunks(section_text: str, section_key: str, chunk_token_size: int, chunks_per_trunk: int):
    """
    将 section 文本切成 chunks，再按 chunks_per_trunk 一组放入 trunks。
    返回 {
        "trunks": [
            { "chunks": [ { "text": "...", "ref": { "section", "trunk_idx", "chunk_idx", "start_token", "end_token" } } ] }
        ]
    }
    """
    raw_chunks = split_into_chunks(section_text, chunk_token_size)
    trunks = []
    for i in range(0, len(raw_chunks), chunks_per_trunk):
        trunk_chunks = raw_chunks[i : i + chunks_per_trunk]
        chunks_list = []
        for j, (ctext, stok, etok) in enumerate(trunk_chunks):
            chunks_list.append({
                "text": ctext,
                "ref": {
                    "section": section_key,
                    "trunk_idx": len(trunks),
                    "chunk_idx": j,
                    "start_token": stok,
                    "end_token": etok,
                },
            })
        trunks.append({"chunks": chunks_list})
    return {"trunks": trunks}


# 小测
sample = " ".join(["word"] * 400)
chunks = split_into_chunks(sample, 256)
print("示例：400 词 → 约", len(chunks), "段，首段 token 区间:", chunks[0][1], "-", chunks[0][2] if chunks else "-")

示例：400 词 → 约 3 段，首段 token 区间: 0 - 255


---
## 步骤 3：加载已解析 JSON 并生成 chunked 结构

遍历 `parsed_filings/*.json`（排除已包含 `_chunked` 的），对每个文件的 `sections` 做细分为 trunks/chunks，写回 `chunked/{basename}_chunked.json`。

In [7]:
def chunk_filing(data: dict, chunk_token_size: int, chunks_per_trunk: int) -> dict:
    """将 data['sections'] 从 { section_key: text } 转为 { section_key: { trunks: [ { chunks: [...] } ] } }。"""
    meta = data.get("meta_data", {})
    info = data.get("processing_info", {})
    sections_raw = data.get("sections", {})
    sections_new = {}
    for sec_key, sec_text in sections_raw.items():
        if not (sec_text and sec_text.strip()):
            sections_new[sec_key] = {"trunks": []}
            continue
        sections_new[sec_key] = build_section_trunks_chunks(
            sec_text, sec_key, chunk_token_size, chunks_per_trunk
        )
    return {
        "meta_data": meta,
        "sections": sections_new,
        "full_text": data.get("full_text", []),
        "processing_info": {
            **(info or {}),
            "chunk_token_size": chunk_token_size,
            "chunks_per_trunk": chunks_per_trunk,
        },
    }


files = [f for f in os.listdir(PARSED_DIR) if f.endswith(".json") and "_chunked" not in f]
if not files:
    print("未找到可处理的 JSON，请先运行 01_simple_parse_sec_filing 并生成 parsed_filings/*.json")
else:
    for fn in files:
        path = os.path.join(PARSED_DIR, fn)
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        out = chunk_filing(data, CHUNK_TOKEN_SIZE, CHUNKS_PER_TRUNK)
        base = fn.replace(".json", "")
        out_path = os.path.join(OUTPUT_DIR, f"{base}_chunked.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)
        print("已写入:", out_path)
    print("完成。")

已写入: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/US/parsed_filings/chunked/INTC_10K_2024-01-26_chunked.json
已写入: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/US/parsed_filings/chunked/NVDA_10K_2024-02-21_chunked.json
完成。


---
## 步骤 4：查看某个 section 的 trunk/chunk 结构示例

读入一份 chunked 结果，打印某个 section 的 trunk 数、每 trunk 的 chunk 数，以及第一个 chunk 的 `ref` 和文本前 200 字。

In [8]:
# 任选一份 chunked 结果查看结构
chunked_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith("_chunked.json")]
sample_file = os.path.join(OUTPUT_DIR, chunked_files[0]) if chunked_files else None
if sample_file and os.path.isfile(sample_file):
    with open(sample_file, "r", encoding="utf-8") as f:
        sample_data = json.load(f)
    secs = sample_data.get("sections", {})
    first_key = list(secs.keys())[0] if secs else None
    if first_key:
        st = secs[first_key]
        trunks = st.get("trunks", [])
        print("Section:", first_key[:70] + ("..." if len(first_key) > 70 else ""))
        print("Trunk 数:", len(trunks))
        if trunks:
            c0 = trunks[0].get("chunks", [])
            print("第 1 个 trunk 的 chunk 数:", len(c0))
            if c0:
                print("第 1 个 chunk ref:", c0[0].get("ref"))
                print("第 1 个 chunk 文本（前 200 字）:", (c0[0].get("text") or "")[:200])
else:
    print("未找到 chunked 文件，请先运行步骤 3。")

Section: Availability of Company Information 2
Trunk 数: 1
第 1 个 trunk 的 chunk 数: 1
第 1 个 chunk ref: {'section': 'Availability of Company Information 2', 'trunk_idx': 0, 'chunk_idx': 0, 'start_token': 0, 'end_token': 233}
第 1 个 chunk 文本（前 200 字）: Availability of Company Information We use our Investor Relations website, www.intc.com , as a routine channel for distribution of important, and often material, information about us, including our qu
